# Accuracy Lab: Foundation Tuning
This notebook is for high-precision experimentation with model outputs. Use this to find the "Golden Configuration" for your extraction prompts.

### Accuracy Levers available here:
1. **Decoding Parameters:** Adjust `temperature` and `top_p`.
2. **Few-Shot Examples:** Provide the model with perfect examples of input -> output.
3. **Context Engineering:** Strip useless noise from OCR text before sending to LLM.

In [ ]:
import time
import json
import re
from openai import OpenAI
from paddleocr import PaddleOCRVL

# Assumes vLLM servers are running on 8000 and 8001
vlm = PaddleOCRVL(vl_rec_backend="vllm-server", vl_rec_server_url="http://localhost:8000/v1", vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL")
llm_client = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")
LLM_MODEL = "Qwen/Qwen3-4B-AWQ" # Update if you use a different model
print("Engines Initialized.")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/teamspace/studios/this_studio/.paddlex/official_models/PP-DocLayoutV3`.
Creating model: ('PaddleOCR-VL-1.5-0.9B', None)


Engines Initialized.


## Step 1: Vision Extraction (The Raw Input)
Change the `IMAGE_PATH` to any document you want to test.

In [ ]:
IMAGE_PATH = "../invoices/invoice007.jpg"

res = vlm.predict(IMAGE_PATH)

[2026-06-03 04:48:12,196] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,209] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,243] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,263] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,276] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,302] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,321] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-06-03 04:48:12,348] [    INFO] _client.py:1740 - HTTP Re

In [ ]:
res

In [ ]:
def extract_and_combine_content(data):
    """Helper to extract content from PaddleOCRVL results."""
    combined_content = []
    if isinstance(data, list) and data:
        if 'parsing_res_list' in data[0] and isinstance(data[0]['parsing_res_list'], list):
            for item in data[0]['parsing_res_list']:
                content = None
                if hasattr(item, 'content'):
                    content = item.content
                elif isinstance(item, dict):
                    content = item.get('content')
                
                if content is not None:
                    combined_content.append(content)
    return '\n'.join(combined_content)


ocr_text = extract_and_combine_content(res)
print("--- RAW OCR TEXT ---")
print(ocr_text)
# print(ocr_text[:500] + "...")

--- RAW OCR TEXT ---
treats BY ANNA MARIE
INVOICE #8934 945
Matthew John Robertson +90 834 8436 +mrobertson@email.co
ISSUED ON 18 NOVEMBER 2019
<table><tr><td>DESCRIPTION</td><td>QTY</td><td>PRICE</td><td>TOTAL</td></tr><tr><td>Vanilla Cupcakes</td><td>40</td><td>$50.00</td><td>$100.00</td></tr><tr><td>Red Velvet Cupcakes</td><td>40</td><td>$35.00</td><td>$35.00</td></tr><tr><td>Chocolate Chip Cookies</td><td>120</td><td>$140.00</td><td>$140.00</td></tr></table>
PAYMENT DETAILS
THANK YOU!
Account Name: Anna Marie Williams  
Account Number: 1920 1804 7293 4983
Treats by Anna Marie +90 8435 • treatsbyannamarie.co


## Step 2: Context Engineering (Noise Filtering)
Experiment with stripping out headers, footers, or repetitive text that might confuse the model.

In [ ]:
# def clean_context(text):
#     # Remove <img> tags from PaddleOCR
#     text = re.sub(r'<img[^>]*>', '', text)
#     # Add your own custom filters here (e.g., stripping long legal disclaimers)
#     return text.strip()

# cleaned_ocr = clean_context(ocr_text)
# print(cleaned_ocr)

In [ ]:
from bs4 import BeautifulSoup

def html_table_to_markdown(html_content):
    """Converts HTML <table> to Markdown table using BeautifulSoup."""
    soup = BeautifulSoup(html_content, 'html.parser')
    tables = soup.find_all('table')
    
    markdown_tables = []
    for table in tables:
        rows = table.find_all('tr')
        if not rows: continue
        
        md_rows = []
        for i, row in enumerate(rows):
            cols = row.find_all(['td', 'th'])
            cols_text = [c.get_text(strip=True) for c in cols]
            md_rows.append("| " + " | ".join(cols_text) + " |")
            
            # Add separator after header
            if i == 0:
                md_rows.append("| " + " | ".join(["---"] * len(cols)) + " |")
        
        markdown_tables.append("\n".join(md_rows))
    
    return "\n\n".join(markdown_tables) if markdown_tables else html_content



def clean_ocr_text(text):
    """Cleans OCR text: removes img tags and converts tables."""
    # 1. Remove <img ...> tags
    text = re.sub(r'<img[^>]*>', '', text)
    
    # 2. Extract <table> contents and convert to markdown
    def table_replacer(match):
        return html_table_to_markdown(match.group(0))
    
    cleaned_text = re.sub(r'<table>.*?</table>', table_replacer, text, flags=re.DOTALL)
    
    # 3. Clean up excessive whitespace
    cleaned_text = re.sub(r'\n\s*\n', '\n\n', cleaned_text)
    
    return cleaned_text.strip()

cleaned_text = clean_ocr_text(ocr_text)
print(cleaned_text)

treats BY ANNA MARIE
INVOICE #8934 945
Matthew John Robertson +90 834 8436 +mrobertson@email.co
ISSUED ON 18 NOVEMBER 2019
| DESCRIPTION | QTY | PRICE | TOTAL |
| --- | --- | --- | --- |
| Vanilla Cupcakes | 40 | $50.00 | $100.00 |
| Red Velvet Cupcakes | 40 | $35.00 | $35.00 |
| Chocolate Chip Cookies | 120 | $140.00 | $140.00 |
PAYMENT DETAILS
THANK YOU!
Account Name: Anna Marie Williams  
Account Number: 1920 1804 7293 4983
Treats by Anna Marie +90 8435 • treatsbyannamarie.co


## Step 3: Few-Shot Prompt Lab
Add examples inside the `FEW_SHOT_EXAMPLES` block. The format should be:
Input: [OCR TEXT]
Output: [EXPECTED JSON]

In [ ]:
FEW_SHOT_EXAMPLES = """
Example 1:
Input: Invoice #99 Vendor Acme Corp Total $100
Output: {"invoice_number": "99", "vendor": "Acme Corp", "total": 100.0}
"""

SYSTEM_PROMPT = """You are a precise data extraction assistant specialized in financial documents.
Extract information from the provided text and return it strictly as a JSON object.

RULES:
1. DATES: All dates MUST be converted to YYYY-MM-DD format.
2. NUMBERS: Convert currency and quantities to float numbers (e.g., ,200.50 -> 1200.50).
3. NULLS: If a field is not present in the text, use null.
4. NESTING: Follow the exact nested structure provided below to separate Vendor vs Client details.
5. LINE ITEMS: Extract every row from tables into the line_items array.
6. BE INTELLIGENT with the Vendor & Client details. Vendor usually comes first with a company name and address. Client usually comes after the vendor section with the invoice number, and usually not same as the vendor. Unless explicitely specified.

HINTS:
- Information that are close together most likely represent the same entity, such as Vendor OR Client
- Vendor Details usually comes first together with the company details
- Vendor and Client are usually not the same, look for different entities at the start of the extracted text.
- Names of Vendors and Clients must be specifically like a name, and not contain any additional words like (by, from, of etc.)
- Be precise in ensuring whether a Proper Nouns is actually a company name or the brand name.
- Vendor and Client cannot be the same
- Handle subitems in a row or a value seperately
- "float" example : 0.00
- address : is location address, not digital email address or any other.

TARGET JSON SCHEMA:
{
  "document_details": {
    "document_type": "string",
    "invoice_number": "string",
    "invoice_date": "YYYY-MM-DD",
    "due_date": "YYYY-MM-DD"
  },
  "vendor_details": {
    "company_name": "string",
    "person_name": "string",
    "address": "string",
    "contact_info": "string"
  },
  "client_details": {
    "company_name": "string",
    "person_name": "string",
    "address": "string",
    "contact_info": "string"
  },
  "line_items": [
    {
      "description": "string",
      "quantity": "float",
      "unit_price": "float",
      "line_total": "float"
    }
  ],
  "financials": {
    "subtotal": "float",
    "tax_amount": "float",
    "total_amount": "float"
  }
}

Extract the following text:"""



# {FEW_SHOT_EXAMPLES}

## Step 4: Execution & Parameter Testing
Adjust `temperature` and `top_p` to see how they impact hallucinations.

In [ ]:
response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": cleaned_text}
    ],
    temperature=0.1, # LEVER 1
    # top_p=0.1,       # LEVER 2
    stream=True, # <-- Added this to enable streaming
    response_format={"type": "json_object"}
)

print(f"\n\n--- LLM EXTRACTION {IMAGE_PATH} ---")
# print(response.choices[0].message.content)

# Loop through the stream and print each chunk as it arrives
for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

[2026-05-29 09:30:46,043] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"




--- LLM EXTRACTION ../invoices/invoice007.jpg ---
{
  "document_details": {
    "document_type": "invoice",
    "invoice_number": "8934 945",
    "invoice_date": "2019-11-18",
    "due_date": null
  },
  "vendor_details": {
    "company_name": "Treats by Anna Marie",
    "person_name": "Anna Marie",
    "address": "treatsbyannamarie.co",
    "contact_info": "+90 8435"
  },
  "client_details": {
    "company_name": "Anna Marie Williams",
    "person_name": null,
    "address": null,
    "contact_info": "1920 1804 7293 4983"
  },
  "line_items": [
    {
      "description": "Vanilla Cupcakes",
      "quantity": 40,
      "unit_price": 50,
      "line_total": 100
    },
    {
      "description": "Red Velvet Cupcakes",
      "quantity": 40,
      "unit_price": 35,
      "line_total": 35
    },
    {
      "description": "Chocolate Chip Cookies",
      "quantity": 120,
      "unit_price": 140,
      "line_total": 140
    }
  ],
  "financials": {
    "subtotal": 275,
    "tax_amount": nul

In [ ]:
response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": cleaned_text}
    ],
    temperature=0.1, # LEVER 1
    # top_p=0.1,       # LEVER 2
    stream=True, # <-- Added this to enable streaming
    response_format={"type": "json_object"}
)

print(f"\n\n--- LLM EXTRACTION {IMAGE_PATH} ---")
# print(response.choices[0].message.content)

# Loop through the stream and print each chunk as it arrives
for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

[2026-05-29 09:23:47,893] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"




--- LLM EXTRACTION ../invoices/invoice007.jpg ---
{
  "document

_details": {
    "document_type": "invoice",
    "invoice_number": "8934 945",
    "invoice_date": "2019-11-18",
    "due_date": null
  },
  "vendor_details": {
    "company_name": "Treats by Anna Marie",
    "person_name": "Anna Marie",
    "address": "treatsbyannamarie.co",
    "contact_info": "+90 8435"
  },
  "client_details": {
    "company_name": "Anna Marie Williams",
    "person_name": null,
    "address": null,
    "contact_info": "1920 1804 7293 4983"
  },
  "line_items": [
    {
      "description": "Vanilla Cupcakes",
      "quantity": 40,
      "unit_price": 50,
      "line_total": 100
    },
    {
      "description": "Red Velvet Cupcakes",
      "quantity": 40,
      "unit_price": 35,
      "line_total": 35
    },
    {
      "description": "Chocolate Chip Cookies",
      "quantity": 120,
      "unit_price": 140,
      "line_total": 140
    }
  ],
  "financials": {
    "subtotal": 275,
    "tax_amount": null,
    "total_amount": 275
  }
}

invoice 005 - got 12% as tax amount , but actual system calulated the value (where is the math happening in system code?)

In [ ]:
response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": cleaned_text}
    ],
    temperature=0.1, # LEVER 1
    # top_p=0.1,       # LEVER 2
    stream=True, # <-- Added this to enable streaming
    response_format={"type": "json_object"}
)

print(f"\n\n--- LLM EXTRACTION {IMAGE_PATH} ---")
# print(response.choices[0].message.content)

# Loop through the stream and print each chunk as it arrives
for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

[2026-05-26 07:59:44,455] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"




--- LLM EXTRACTION ../invoices/invoice006.jpg ---
{
  "document_details": {
    "document_type": "invoice",
    "invoice_number": "760 982 11",
    "invoice_date": null,
    "due_date": null
  },
  "vendor_details": {
    "company_name": "COOKIES",
    "person_name": "BY CHARLIE",
    "address": "260 Quad Oak Dr, Mount Juliet, TN, 37122",
    "contact_info": null
  },
  "client_details": {
    "company_name": null,
    "person_name": "MS. OLIVIA W. ADAMS",
    "address": "246 5th Ave. New York, NY",
    "contact_info": "246 5th Ave. New York, NY"
  },
  "line_items": [
    {
      "description": "Chocolate Chip",
      "quantity": 12,
      "unit_price": 5,
      "line_total": 60
    },
    {
      "description": "Taffee Crunch",
      "quantity": 12,
      "unit_price": 5,
      "line_total": 60
    },
    {
      "description": "Peanut Butter Combo",
      "quantity": 6,
      "unit_price": 5,
      "line_total": 30
    },
    {
      "description": "Cinnamon Butter",
      "quanti